In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.coordinates as coord
import astropy.units as u
import pandas as pd
import sys
from scipy.special import factorial
from astropy.table import Table
from numpy.polynomial.polynomial import Polynomial
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib

if './SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('./SelfCalGroupFinder/py/')
from pyutils import *
import plotting as pp
from dataloc import *
from bgs_helpers import *
import catalog_definitions as cat
from groupcatalog import *

%load_ext autoreload
%autoreload 2

BOX_SIZE_MPC_H = 400  # Mpc/h, from the smdpl documentation
BOX_VOLUME = BOX_SIZE_MPC_H**3  # (Mpc/h)^3

# Define file paths
directory = '/mount/sirocco1/imw2293/GROUP_CAT/DATA/POPMOCK/'
file = directory + 'smdpl_z0.19717.M1E10.h5'

In [ ]:
#table = create_merged_file(BGS_Y3_ANY_FULL_FILE, IAN_BGS_Y3_MERGED_FILE_LOA, "3")
# Read the entire HDF5 file into memory
df = pd.read_hdf(file, key='halos')
df = df[df['upid'] == -1] # Keep only central halos

In [ ]:
# Compare M200b to Mpeak
x = np.log10(df['M200b'])
y = np.log10(df['Mpeak'])

fig, ax = plt.subplots(figsize=(8, 6))

# 2D histogram as the basis for contours
h, xedges, yedges = np.histogram2d(x, y, bins=100)
xcenters = 0.5 * (xedges[:-1] + xedges[1:])
ycenters = 0.5 * (yedges[:-1] + yedges[1:])

contour = ax.contourf(xcenters, ycenters, h.T, levels=20, cmap='viridis')
fig.colorbar(contour, ax=ax, label='Count')
ax.contour(xcenters, ycenters, h.T, levels=20, colors='k', linewidths=0.3, alpha=0.4)

ax.set_xlabel('log10(M200b)')
ax.set_ylabel('log10(Mpeak)')
ax.set_title('M200b vs Mpeak for Central Halos')
ax.grid(True, ls="--", lw=0.5, alpha=0.5)
plt.show()

In [ ]:
# I think rvir is OK for this c calculation
df['c'] = df['rvir'] / df['rs'] # both in kpc/h, so this is dimensionless
df['LOGMHALO'] = np.log10(df['M200b']) # Mpeak instead of M200b? Sometimes it way higher for some reason. Keep M200b
df['NOISE'] = np.random.normal(0, 0.1, size=len(df)) # For testing PCA with pure noise noisy feature

In [ ]:
# Seperate into test/training sets
from sklearn.model_selection import train_test_split
train, test = train_test_split(df, test_size=0.1, random_state=894)

feature_cols = ['LOGMHALO', 'c', 'Spin', 'Halfmass_Scale']
n_features = len(feature_cols) # No dimensionality reduction right now. 
# If we add more properties and want to reduce dimensionality, we can change this, but we should see if the recovery tests below still pass and decide what is OK for us.

# Drop rows with any NaN in the selected features
train_clean = train.dropna(subset=feature_cols)
test_clean = test.dropna(subset=feature_cols)
print(f"Training samples after NaN drop: {len(train_clean)} / {len(train)}")
print(f"Test samples after NaN drop: {len(test_clean)} / {len(test)}")

In [ ]:
# Scale first - critical since features have very different units/ranges
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_clean[feature_cols])
test_scaled = scaler.transform(test_clean[feature_cols])

In [ ]:
# Compare the original components to the scaled ones
for f in feature_cols:
    plt.hist(train_clean[f], bins=100, alpha=0.5, label=f'Original {f}')
    plt.hist(train_scaled[:, feature_cols.index(f)], bins=100, alpha=0.5, label=f'Scaled {f}')
    plt.legend()
    plt.title(f'Original vs Scaled {f}')
    plt.yscale('log')
    plt.show()

In [ ]:
# Fit full PCA on training set
pca_full = PCA(n_components=n_features)
pca_full.fit(train_scaled)

# Explained variance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1, n_features+1), pca_full.explained_variance_ratio_, color='steelblue')
axes[0].set_xlabel('PCA Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('Variance per Component')

axes[1].plot(range(1, n_features+1), np.cumsum(pca_full.explained_variance_ratio_), 'ko-')
axes[1].axhline(0.95, color='red', linestyle='--', label='95%')
axes[1].axhline(0.99, color='orange', linestyle='--', label='99%')
axes[1].set_xlabel('Number of PCA Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Print feature loadings for top components
print("\nFeature loadings for top 4 components:")
loadings = pd.DataFrame(
    pca_full.components_[:4].T,
    index=feature_cols,
    columns=[f'PC{i+1}' for i in range(4)]
)
print(loadings.round(3).to_string())

In [ ]:
# Apply PCA transformation and join the columns
train_pca_full = pca_full.transform(train_scaled)
test_pca_full = pca_full.transform(test_scaled)
train_with_pca = pd.concat([train_clean.reset_index(drop=True), pd.DataFrame(train_pca_full, columns=[f'PCA{i+1}' for i in range(train_pca_full.shape[1])])], axis=1)
test_with_pca = pd.concat([test_clean.reset_index(drop=True), pd.DataFrame(test_pca_full, columns=[f'PCA{i+1}' for i in range(test_pca_full.shape[1])])], axis=1)

In [ ]:
full_with_pca = pd.concat([train_with_pca, test_with_pca], ignore_index=True)

In [ ]:
# Save off the model and scaler for later use
# TODO might need this format to be something C++ can also read. Just some matrices anyway, right?
joblib.dump((pca_full, scaler, feature_cols), HALO_PCA_MODEL_FILE)

In [ ]:
# Save the density functions

from scipy import stats

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlabel('PCAx Value')
ax.set_yscale('log')
ax.set_ylabel('Density [1/(Mpc/h)^3]')
ax.grid(True, ls="--", lw=0.5, alpha=0.5)

for i in range(train_pca_full.shape[1]):
    colname = f'PCA{i+1}'
    data = full_with_pca[colname].values

    bins = np.linspace(-15, 15, 200)
    # Could make bins wider on edges? Will the C++ spline fitting work?
    dPCAx = np.diff(bins)[0]
    counts, _ = np.histogram(data, bins=bins)
    pca_centers = 0.5 * (bins[:-1] + bins[1:])
    density = counts / dPCAx / BOX_VOLUME
    output_data = np.column_stack((pca_centers, density))

    c = pp.get_color(i)
    ax.plot(pca_centers, density, drawstyle='steps-mid', label=f'{colname}', color=c)

    # C++ spline fitting may not be able to capture these shapes, unlike the HMF which is easy.

    if i == 0:   np.savetxt(HALO_PCA1_DENSITY_FUNC_FILE, output_data)
    elif i == 1: np.savetxt(HALO_PCA2_DENSITY_FUNC_FILE, output_data)
    elif i == 2: np.savetxt(HALO_PCA3_DENSITY_FUNC_FILE, output_data)
    elif i == 3: np.savetxt(HALO_PCA4_DENSITY_FUNC_FILE, output_data)

ax.legend()
plt.tight_layout()
plt.show()

### Tests

In [ ]:
# Let's test if the rockstar halos we have here roughly match the HMF from Tinker08
# There will be minor cosmology difference but doesn't matter.
# They seem to match, so all is well.

hmf = pd.read_csv(HMF_T08_P18_FILE, delim_whitespace=True, names=['Mh', 'dn/dM'])

# Compute dn/dM from the simulation halos
mbins = np.logspace(10, 16, 60)  # M1E10 cut means nothing below 1e10
mcenters = 0.5 * (mbins[:-1] + mbins[1:])
dM = np.diff(mbins)

counts, _ = np.histogram(df['M200b'], bins=mbins)
dn_dM_sim = counts / dM / BOX_VOLUME

# Mask out empty bins
nonempty = counts > 0

plt.figure(figsize=(7, 5))
plt.plot(hmf['Mh'], hmf['dn/dM'], label='Tinker08 HMF (Planck18)', color='steelblue', lw=2)
plt.plot(mcenters[nonempty], dn_dM_sim[nonempty], 'o', ms=4, color='r', label='SMDPL Rockstar centrals')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r'$M_h \ (M_\odot h^{-1})$')
plt.ylabel(r'dn/dM $((M_\odot h^{-1})^{-1} \, \mathrm{Mpc}^{-3} h^3)$')
plt.title('Simulation HMF vs Tinker08 (Planck18)')
plt.grid(True, ls="--", lw=0.5, alpha=0.5)
plt.legend()
plt.tight_layout()
plt.show()

# Quantitative check: relative difference in the well-sampled mass range
from scipy.interpolate import interp1d
hmf_interp = interp1d(hmf['Mh'], hmf['dn/dM'], bounds_error=False, fill_value=np.nan)
expected = hmf_interp(mcenters[nonempty])
rel_diff = (dn_dM_sim[nonempty] - expected) / expected
print(f"Median relative difference vs Tinker08: {np.nanmedian(rel_diff)*100:.1f}%")
print(f"Max relative difference vs Tinker08:    {np.nanmax(np.abs(rel_diff))*100:.1f}%")

In [ ]:
# Roundtrip PCA test
testpoints = 10000
sample = test_with_pca[feature_cols + [f'PCA{i+1}' for i in range(n_features)]].iloc[:testpoints]
recovered = halo_pca_to_original(sample[[f'PCA{i+1}' for i in range(n_features)]].values)
tol = 1e-5 # more than really needed
assert np.allclose(sample[feature_cols].values, recovered.values, atol=tol), "Roundtrip PCA transformation did not recover original values within tolerance!"

In [ ]:
# See how well the distributions (including some conditinoal checks) are recovered.
# Since we do no dim reduction, this is really just checking if the scaling and inverse scaling are working correctly.
# Everything should be 'perfectly' recovered up to numerical precision.
# If we add dim reduction later, we can decide what level of distribution mismatch is acceptable for our use case.

from scipy.stats import ks_2samp # KS test is a goodness of fit test for 2 distributions
pca_cols = [f'PCA{i+1}' for i in range(n_features)]
test_recovered = halo_pca_to_original(test_with_pca[pca_cols].values)

# --- 1. Marginal distributions: KS test for each feature ---
print("=== Marginal distribution KS tests ===")
for col in feature_cols:
    # For Halfmass_Scale, use max absolute error instead of KS (bounded distribution with sharp edge)
    if col == 'Halfmass_Scale':
        max_err = np.abs(test_with_pca[col].values - test_recovered[col].values).max()
        print(f"  {col:20s}  max abs error={max_err:.2e}  (KS skipped: bounded distribution)")
        assert max_err < 1e-4, f"Max error for {col} too large: {max_err}"
    else:
        stat, p = ks_2samp(test_with_pca[col].values, test_recovered[col].values)
        print(f"  {col:20s}  KS stat={stat:.2e}  p={p:.4f}")
        assert p > 0.99, f"Marginal distribution of {col} not recovered (p={p:.4f})"

# --- 2. Means and stds match ---
print("\n=== Mean and std recovery ===")
for col in feature_cols:
    orig_mean, rec_mean = test_with_pca[col].mean(), test_recovered[col].mean()
    orig_std,  rec_std  = test_with_pca[col].std(),  test_recovered[col].std()
    print(f"  {col:20s}  mean: {orig_mean:.4f} vs {rec_mean:.4f}  |  std: {orig_std:.4f} vs {rec_std:.4f}")
    assert np.isclose(orig_mean, rec_mean, rtol=1e-4), f"Mean mismatch for {col}"
    assert np.isclose(orig_std,  rec_std,  rtol=1e-4), f"Std mismatch for {col}"

# --- 3. Conditional distributions in halo mass bins ---
print("\n=== Conditional distribution KS tests (c | LOGMHALO bin) ===")
mass_bins = np.percentile(test_with_pca['LOGMHALO'], [0, 33, 66, 100])
for lo, hi in zip(mass_bins[:-1], mass_bins[1:]):
    mask = (test_with_pca['LOGMHALO'] >= lo) & (test_with_pca['LOGMHALO'] < hi)
    orig_c = test_with_pca.loc[mask, 'c'].values
    rec_c  = test_recovered.loc[mask, 'c'].values
    stat, p = ks_2samp(orig_c, rec_c)
    print(f"  LOGMHALO [{lo:.2f}, {hi:.2f})  n={mask.sum()}  KS stat={stat:.2e}  p={p:.4f}")
    assert p > 0.99, f"Conditional c distribution not recovered in mass bin [{lo:.2f}, {hi:.2f})"

# --- 4. Conditional: Spin | concentration bin ---
print("\n=== Conditional distribution KS tests (Spin | c bin) ===")
c_bins = np.percentile(test_with_pca['c'], [0, 50, 100])
for lo, hi in zip(c_bins[:-1], c_bins[1:]):
    mask = (test_with_pca['c'] >= lo) & (test_with_pca['c'] < hi)
    orig_s = test_with_pca.loc[mask, 'Spin'].values
    rec_s  = test_recovered.loc[mask, 'Spin'].values
    stat, p = ks_2samp(orig_s, rec_s)
    print(f"  c [{lo:.2f}, {hi:.2f})  n={mask.sum()}  KS stat={stat:.2e}  p={p:.4f}")
    assert p > 0.99, f"Conditional Spin distribution not recovered in c bin [{lo:.2f}, {hi:.2f})"

print("\nAll distribution tests passed.")